# Version 2: direction classification with an auxiliary return forecast
Upload the revised `train.py` and `inference.py` to `/home/sagemaker-user/stock-ml/src/` before running. Run training first and deploy only its successful new artifact. CSV features must already use the original training scaler. LONG/CASH now uses validation-selected positive-return probability, not the sign of the auxiliary return forecast. Existing Lambda signal handling remains compatible.
Validation must contain both signals and balanced accuracy >= 0.505. A failed gate means this run is not suitable for deployment; do not reuse an old model. Evaluate the frozen model on the full untouched test period after selection.


In [ ]:
import boto3
import os
import shutil
import tarfile
import json
import pandas as pd

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.serve.model_builder import ModelBuilder


session = Session()
role = get_execution_role()
region = session.boto_region_name

s3 = boto3.client("s3", region_name=region)

print("Region:", region)
print("Role:", role)

In [ ]:
bucket = "stock-dl-2026"

trained_model_key = "models/ffnn-v2/model.tar.gz"

local_model_tar = "/tmp/model.tar.gz"
extract_dir = "/tmp/model_package"
new_model_tar = "/tmp/model-inference.tar.gz"


# Download existing trained model
s3.download_file(
    bucket,
    trained_model_key,
    local_model_tar
)


# Extract it
shutil.rmtree(extract_dir, ignore_errors=True)
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(local_model_tar, "r:gz") as tar:
    tar.extractall(extract_dir)


# Add updated inference.py
code_dir = os.path.join(extract_dir, "code")
os.makedirs(code_dir, exist_ok=True)

shutil.copy2(
    "/home/sagemaker-user/stock-ml/src/inference.py",
    os.path.join(code_dir, "inference.py")
)


# Rebuild model package
with tarfile.open(new_model_tar, "w:gz") as tar:
    for name in os.listdir(extract_dir):
        tar.add(
            os.path.join(extract_dir, name),
            arcname=name
        )


print("Package contents:")

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        print(
            os.path.relpath(
                os.path.join(root, file),
                extract_dir
            )
        )

In [ ]:
inference_model_key = "models/ffnn-v2/model-inference-v2.tar.gz"

s3.upload_file(
    new_model_tar,
    bucket,
    inference_model_key
)

model_data_uri = (
    f"s3://{bucket}/{inference_model_key}"
)

print("Uploaded:", model_data_uri)

In [ ]:
inference_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.2.0",
    py_version="py310",
    instance_type="ml.m5.xlarge",
    image_scope="inference"
)

print(inference_image)

In [ ]:
MODEL_NAME = "stock-ffnn-direction-v2"

model_builder = ModelBuilder(
    image_uri=inference_image,
    s3_model_data_url=model_data_uri,
    role_arn=role,
    sagemaker_session=session
)

model = model_builder.build(
    model_name=MODEL_NAME
)

print("Model created:", model.get_name())

In [ ]:
FEATURE_COLS = [
    "LogReturn",
    "SMA_5_20_diff",
    "SMA_50_200_diff",
    "RSI",
    "MACD",
    "MACD_Signal",
    "MACD_Histogram",
    "BB_width",
    "BB_pct",
    "ATR_pct",
    "OBV_z",
    "VIX_Close",
]


test_df = pd.read_csv(
    "s3://stock-dl-2026/processed/ml/test/test.csv"
)


# Latest available row for each ticker
latest = (
    test_df
    .sort_values("Date")
    .groupby("Ticker", as_index=False)
    .tail(1)
    .sort_values("Ticker")
)


payload = {
    "tickers": latest["Ticker"].tolist(),
    "features": latest[FEATURE_COLS].values.tolist()
}


with open("/tmp/batch_test.jsonl", "w") as f:
    f.write(json.dumps(payload) + "\n")


print("Number of tickers:", len(payload["tickers"]))
print(payload["tickers"])

In [ ]:
s3.upload_file(
    "/tmp/batch_test.jsonl",
    "stock-dl-2026",
    "demo/inference-ready/batch_test.jsonl"
)

print(
    "Uploaded:",
    "s3://stock-dl-2026/demo/inference-ready/batch_test.jsonl"
)

In [ ]:
# Evaluate the frozen model across the full test period before relying on snapshots.
# This uses the same freshly downloaded checkpoint and inference script packaged above.
import importlib.util
import numpy as np
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

spec = importlib.util.spec_from_file_location("serving_v2", os.path.join(code_dir, "inference.py"))
serving = importlib.util.module_from_spec(spec)
spec.loader.exec_module(serving)
artifact = serving.model_fn(extract_dir)
full_payload = {"tickers": test_df["Ticker"].tolist(),
                "features": test_df[FEATURE_COLS].values.tolist()}
full_results = serving.predict_fn(serving.input_fn(json.dumps(full_payload), "application/json"), artifact)
results = pd.DataFrame(full_results)
actual_up = test_df["target_next_day_logreturn"].to_numpy() > 0
print("Full-test LONG/CASH counts:", results["position"].value_counts().to_dict())
print("Balanced accuracy:", balanced_accuracy_score(actual_up, results["signal"]))
print("Confusion matrix [actual CASH/LONG rows; predicted CASH/LONG columns]:")
print(confusion_matrix(actual_up, results["signal"], labels=[0, 1]))
results.insert(0, "Date", test_df["Date"].to_numpy())
results.to_csv("ffnn_v2_full_test_predictions.csv", index=False)
# Do not tune the threshold or choose seeds based on these test results.
